# Universal Grammar Generative Model

In [21]:
INSTALL = false

if INSTALL
    using Pkg
    Pkg.activate("psyc261")
    Pkg.add(["JLD2", "Distributions", "ProgressMeter", "Parameters", 
            "Random", "Gen", "Plots", "PyCall", "Conda", "JSON3"])
end

In [22]:
include("../src/ug_model.jl")
using .UGmodel

In [23]:
using Pkg
Pkg.activate("psyc261")
using JLD2
using Distributions
using ProgressMeter
using Parameters
using Random
using Gen, Plots
using PyCall
np = pyimport("numpy")

  Activating project at `~/Algorithms-of-the-Mind/aom/notebooks/psyc261`


PyObject <module 'numpy' from '/home/psyc2610_jdp69/.local/lib/python3.11/site-packages/numpy/__init__.py'>

## Generative Model

In [24]:
const NOUNS = UGmodel.LEXICON.nouns
const VERBS_TRANS = UGmodel.LEXICON.verbs_trans
const VERBS_INTR = UGmodel.LEXICON.verbs_intr
const ADJECTIVES = UGmodel.LEXICON.adjectives
const WH_WORD = UGmodel.LEXICON.wh_word

"what"

In [25]:
# Defining a deterministic distribution for strings so it's possible to "observe" them
struct StringDist <: Gen.Distribution{Vector{String}} end
const string_dist = StringDist()

(::StringDist)(val) = val

Gen.logpdf(::StringDist, x, val) = (x == val) ? 0. : -Inf
Gen.random(::StringDist, val) = val

In [26]:
# Noise constant to account for misunderstandings, slip of the tongue etc.
# Generates ungrammatical sentences
const NOISE_PROB = 0.1

0.1

In [27]:
@gen function corpus_model(num_sentences::Int)
    # === Universal Grammar paramaters (theta) ===
    # Head direction: 1 = Verb-Object (VO), 0 = Object-Verb(OV)
    hd = @trace(bernoulli(0.5), :head_direction)

    # Subject drop: 1 = subject may be dropped, 0 = subject always present
    sd = @trace(bernoulli(0.5), :subject_drop)

    # Adjective order: 1 = Adjective-Noun (AN), 0 = Noun-Adjective (NA)
    ao = @trace(bernoulli(0.5), :adj_order)

    # Wh-fronting in question: 1 = wh-word at front, 0 = in-situ
    wh = @trace(bernoulli(0.5), :wh_fronting)
    
    sentences = Vector{Vector{String}}(undef, num_sentences)

    for i in 1:num_sentences
        # Utterance type: 1 = transitive declarative, 2 = intransitive, 3 = transitive question
        utt_type = @trace(categorical([0.4, 0.3, 0.3]), (:utt_type, i))

        # Sample lexical items
        subj_idx = @trace(categorical(fill(1/length(NOUNS), length(NOUNS))), (:subj, i))
        obj_idx = @trace(categorical(fill(1/length(NOUNS), length(NOUNS))), (:obj, i))
        adj_idx = @trace(categorical(fill(1/length(ADJECTIVES), length(ADJECTIVES))), (:adj, i))

        if utt_type == 1 || utt_type == 3
            v_idx = @trace(categorical(fill(1/length(VERBS_TRANS), length(VERBS_TRANS))), (:v_trans, i))
            verb = VERBS_TRANS[v_idx]
        else
            v_idx = @trace(categorical(fill(1/length(VERBS_INTR), length(VERBS_INTR))), (:v_intr, i))
            verb = VERBS_INTR[v_idx]
        end

        subj = NOUNS[subj_idx]
        obj = NOUNS[obj_idx]
        adj = ADJECTIVES[adj_idx]

        # Add adjective order error
        ao_slip = @trace(bernoulli(NOISE_PROB), (:ao_slip, i))
        effective_ao = ao_slip ? !ao : ao
        
        # Build NP for subject and object, respecting adjective order and taking error into account
        function make_np(noun::String, adj::String, order_rule::Bool)
            if order_rule == 1
                return [adj, noun] # Adj N
            else
                return [noun, adj] # N Adj
            end
        end

        subj_np = make_np(subj, adj, effective_ao)
        obj_np = make_np(obj, adj, effective_ao)

        # Add head direction error
        hd_slip = @trace(bernoulli(NOISE_PROB), (:hd_slip, i))
        effective_hd = hd_slip ? !hd : hd

        # Base clause word order for transitive
        vp = Vector{String}()
        if utt_type == 1 || utt_type == 3
            # Verb-Object vs. Object-Verb in the VP
            vp = (effective_hd == 1) ? vcat([verb], obj_np) : vcat(obj_np, [verb])
        else
            # Intransitive: just the verb
            vp = [verb]
        end

        # Add subject drop error
        sd_slip = @trace(bernoulli(NOISE_PROB), (:sd_slip, i))
        effective_sd = sd_slip ? !sd : sd

        # Drop subject if SD = 1
        drop_subj = (effective_sd == 1) ? @trace(bernoulli(0.5), (:drop_subj, i)) : 0

        clause = Vector{String}()

        if drop_subj == 0
            append!(clause, subj_np)
        end
        
        append!(clause, vp)

        # Handle questions
        if utt_type == 3
            wh_slip = @trace(bernoulli(NOISE_PROB), (:wh_slip, i))
            effective_wh = wh_slip ? !wh : wh
            
            if effective_wh == 1
                # wh-fronting
                clause = vcat([WH_WORD], clause)
            else
                # wh in-situ (append at the end)
                push!(clause, WH_WORD)
            end
        end

        sentences[i] = clause

        @trace(string_dist(clause), (:sentence, i))
    end

    return sentences
end

DynamicDSLFunction{Any}(Dict{Symbol, Any}(), Dict{Symbol, Any}(), Type[Int64], false, Union{Nothing, Some{Any}}[nothing], var"##corpus_model#297", Bool[0], false)

In [28]:
trace = simulate(corpus_model, (10,))
sentences = get_retval(trace)
theta = (
    hd = trace[:head_direction],
    sd = trace[:subject_drop],
    ao = trace[:adj_order],
    wh = trace[:wh_fronting]
)
theta

(hd = true, sd = true, ao = false, wh = true)

In [29]:
@gen function corpus_given_theta(hd::Bool, sd::Bool, ao::Bool, wh::Bool, num_sentences::Int)
    sentences = Vector{Vector{String}}(undef, num_sentences)

    for i in 1:num_sentences
        utt_type = @trace(categorical([0.4, 0.3, 0.3]), (:utt_type, i))

        subj_idx = @trace(categorical(fill(1/length(NOUNS), length(NOUNS))), (:subj, i))
        obj_idx = @trace(categorical(fill(1/length(NOUNS), length(NOUNS))), (:obj, i))
        adj_idx = @trace(categorical(fill(1/length(ADJECTIVES), length(ADJECTIVES))), (:adj, i))

        if utt_type == 1 || utt_type == 3
            v_idx = @trace(categorical(fill(1/length(VERBS_TRANS), length(VERBS_TRANS))), (:v_trans, i))
            verb = VERBS_TRANS[v_idx]
        else
            v_idx = @trace(categorical(fill(1/length(VERBS_INTR), length(VERBS_INTR))), (:v_intr, i))
            verb = VERBS_INTR[v_idx]
        end

        subj = NOUNS[subj_idx]
        obj = NOUNS[obj_idx]
        adj = ADJECTIVES[adj_idx]

        function make_np(noun::String, adj::String)
            if ao == 1
                return [adj, noun] # Adj N
            else
                return [noun, adj] # N Adj
            end
        end
        
        subj_np = make_np(subj, adj)
        obj_np = make_np(obj, adj)

        drop_subj = (sd == 1) ? @trace(bernoulli(0.5), (:drop_subj, i)) : 0

        if utt_type == 1 || utt_type == 3
            vp = (hd == 1) ? vcat([verb], obj_np) : vcat(obj_np, [verb])
        else
            vp = [verb]
        end

        clause = Vector{String}()

        if drop_subj == 0
            append!(clause, subj_np)
        end
        append!(clause, vp)

        if utt_type == 3
            if wh == 1
                clause = vcat([WH_WORD], clause)
            else
                push!(clause, WH_WORD)
            end
        end

        sentences[i] = clause
    end

    return sentences
end

DynamicDSLFunction{Any}(Dict{Symbol, Any}(), Dict{Symbol, Any}(), Type[Bool, Bool, Bool, Bool, Int64], false, Union{Nothing, Some{Any}}[nothing, nothing, nothing, nothing, nothing], var"##corpus_given_theta#298", Bool[0, 0, 0, 0, 0], false)

In [30]:
using JSON3

SAVE = false

if SAVE
    theta = (hd=false, sd=false, ao=true, wh=true)
    num_sentences = 100
    
    trace = simulate(corpus_given_theta, (theta.hd, theta.sd, theta.ao, theta.wh, num_sentences))
    corpus = get_retval(trace)
    
    open("../data/ug_language1.json", "w") do io
        JSON3.write(io, Dict("theta" => theta, "sentences" => corpus))
    end
end

## Neural Network

In [31]:
# Building the fixed vocabulary
const VOCAB = unique(vcat(["PAD", WH_WORD], NOUNS, VERBS_TRANS, VERBS_INTR, ADJECTIVES))
const VOCAB_SIZE = length(VOCAB)
const WORD_TO_IDX = Dict(w => i for (i,w) in enumerate(VOCAB))
const MAX_SENT_LEN = 6
const NUM_SENTS_TRAIN = 10
const INPUT_DIM = VOCAB_SIZE * MAX_SENT_LEN * NUM_SENTS_TRAIN

720

### Encoding the sentences

In [32]:
# Convert sentences into a flattened one hot encoded vector
function encode_corpus(sentences::Vector{Vector{String}})
    encoded = zeros(Float64, INPUT_DIM)

    cursor = 0
    for i in 1:NUM_SENTS_TRAIN
        if i > length(sentences) break end

        sent = sentences[i]
        for j in 1:MAX_SENT_LEN
            word = j <= length(sent) ? sent[j] : "PAD"

            vocab_idx = WORD_TO_IDX[word]

            encoded[cursor * VOCAB_SIZE + vocab_idx] = 1.0
            cursor += 1
        end
    end

    return encoded
end

encode_corpus (generic function with 1 method)

### Creating the innate solver neural network

In [33]:
σ(x) = tanh.(x)
sigmoid(x) = 1.0 ./ (1.0 .+ exp.(-x))
H = 128

@gen function innate_solver(input_vector::Vector{Float64})
    @param W1::Matrix{Float64}
    @param b1::Vector{Float64}
    @param W2::Matrix{Float64}
    @param b2::Vector{Float64}

    hidden_layer = σ(W1 * input_vector + b1)
    output = W2 * hidden_layer + b2

    probs = sigmoid(output)

    hd = @trace(bernoulli(probs[1]), :head_direction)
    sd = @trace(bernoulli(probs[2]), :subject_drop)
    ao = @trace(bernoulli(probs[3]), :adj_order)
    wh = @trace(bernoulli(probs[4]), :wh_fronting)

    return nothing
end

init_weight(out, inc) = randn(out, inc) * 0.01
init_param!(innate_solver, :W1, init_weight(H, INPUT_DIM))
init_param!(innate_solver, :b1, zeros(H))
init_param!(innate_solver, :W2, init_weight(4, H))
init_param!(innate_solver, :b2, zeros(4))

4-element Vector{Float64}:
 0.0
 0.0
 0.0
 0.0

### Generating synthetic languages from the GM

In [34]:
function training_batch_generator()
    tr = simulate(corpus_model, (NUM_SENTS_TRAIN, ))

    sentences = get_retval(tr)
    input_vec = encode_corpus(sentences)

    constraints = Gen.choicemap()
    constraints[:head_direction] = tr[:head_direction]
    constraints[:subject_drop] = tr[:subject_drop]
    constraints[:adj_order] = tr[:adj_order]
    constraints[:wh_fronting] = tr[:wh_fronting]

    return (input_vec,), constraints
end

training_batch_generator (generic function with 1 method)

### Training the innate solver

In [ ]:
update = Gen.ParamUpdate(Gen.FixedStepGradientDescent(0.001), innate_solver)

NUM_EPOCHS = 200
EPOCH_SIZE = 50

TRAIN = true

if TRAIN
    scores = Gen.train!(
        innate_solver, 
        training_batch_generator, 
        update, 
        num_epoch = NUM_EPOCHS,
        epoch_size = EPOCH_SIZE,
        num_minibatch = 10,
        minibatch_size = 5,
        verbose = true
    )

    plot(scores, label = "Loss (Neg Elbo)", xlabel = "Epochs", title = "Training innate solver")
end

epoch 1: generating 50 training examples...
epoch 1: training using 10 minibatches of size 5...
epoch 1: evaluating on 50 examples...
epoch 1: est. objective value: 0.0
epoch 2: generating 50 training examples...
epoch 2: training using 10 minibatches of size 5...
epoch 2: evaluating on 50 examples...
epoch 2: est. objective value: 0.0
epoch 3: generating 50 training examples...
epoch 3: training using 10 minibatches of size 5...
epoch 3: evaluating on 50 examples...
epoch 3: est. objective value: 0.0
epoch 4: generating 50 training examples...
epoch 4: training using 10 minibatches of size 5...
epoch 4: evaluating on 50 examples...
epoch 4: est. objective value: 0.0
epoch 5: generating 50 training examples...
epoch 5: training using 10 minibatches of size 5...
epoch 5: evaluating on 50 examples...
epoch 5: est. objective value: 0.0
epoch 6: generating 50 training examples...
epoch 6: training using 10 minibatches of size 5...
epoch 6: evaluating on 50 examples...
epoch 6: est. objecti

## Comparison Innate Solver vs Naive Learner

In [ ]:
target_theta = (hd=true, sd=false, ao=true, wh=true)

target_constraints = Gen.choicemap()
target_constraints[:head_direction] = target_theta.hd
target_constraints[:subject_drop] = target_theta.sd
target_constraints[:adj_order] = target_theta.ao
target_constraints[:wh_fronting] = target_theta.wh

tr_target, _ = generate(corpus_model, (NUM_SENTS_TRAIN,), target_constraints)
observed_sentences = get_retval(tr_target)
observed_input_vec = encode_corpus(observed_sentences)

function mh_step(tr)
    tr, _ = mh(tr, select(:head_direction))
    tr, _ = mh(tr, select(:subject_drop))
    tr, _ = mh(tr, select(:adj_order))
    tr, _ = mh(tr, select(:wh_fronting))
    return tr
end

function is_correct(tr)
    return tr[:head_direction] == target_theta.hd &&
           tr[:subject_drop] == target_theta.sd &&
           tr[:adj_order] == target_theta.ao &&
           tr[:wh_fronting] == target_theta.wh
end

naive_trace, _ = generate(corpus_model, (NUM_SENTS_TRAIN, ), Gen.choicemap())
naive_accuracy = []
for i in 1:500
    global naive_trace = mh_step(naive_trace)
    push